In [ ]:
import pandas as pd
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import joblib

# pip install pyarrow

In [2]:
# Definir ruta donde estan los archivos
path = r'../data/raw/'

In [3]:
# Listar archivos Parquet que comienzan con 'data'
all_files = glob.glob(os.path.join(path, 'data*.parquet'))

In [4]:
# Leer cada archivo Parquet y almacenarlo en una lista de DataFrames
df_list = []
for filename in all_files:
    df_part = pd.read_parquet(filename, engine='pyarrow')
    df_list.append(df_part)

In [5]:
# Concatenar todos los DataFrames en un único DataFrame
df = pd.concat(df_list, axis=0, ignore_index=True)

In [6]:
# Convertimos Date a datetime (necesario para split temporal)
df['Date'] = pd.to_datetime(df['Date'])

C:\Users\Usuario\AppData\Local\Temp\ipykernel_26868\1063807796.py:2: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['Date'] = pd.to_datetime(df['Date'])


1.1 Feature 1 — Rendimiento 5 días

In [ ]:
df['Price_Change_5d']


0        -0.052579
1        -0.031547
2        -0.046896
3        -0.010438
4         0.026642
            ...   
620090    0.002410
620091    0.001953
620092    0.028600
620093    0.015769
620094    0.005231
Name: Price_Change_5d, Length: 620095, dtype: float64

1.2 Feature 2 — Precio vs media 20 días

In [8]:
df['price_vs_sma20'] = (df['Close'] / df['SMA_20']) - 1


1.3 Feature 3 — Fuerza del movimiento (RSI categórico)

In [9]:
def rsi_strength(rsi):
    if rsi < 40:
        return 0   # Débil
    elif rsi <= 60:
        return 1   # Neutral
    else:
        return 2   # Fuerte

df['rsi_strength'] = df['RSI'].apply(rsi_strength)


1.4 Feature 4 — Volumen relativo

In [ ]:
df['Volume_Ratio']

0         0.869463
1         0.681483
2         1.350726
3         0.950337
4         0.831386
            ...   
620090    0.865129
620091    0.961570
620092    1.397328
620093    1.497178
620094    2.580893
Name: Volume_Ratio, Length: 620095, dtype: float64

1.5 Feature 5 — Volatilidad categórica

In [11]:
v1 = df['Volatility'].quantile(0.33)
v2 = df['Volatility'].quantile(0.66)

def vol_level(v):
    if v < v1:
        return 0   # Baja
    elif v < v2:
        return 1   # Media
    else:
        return 2   # Alta

df['vol_level'] = df['Volatility'].apply(vol_level)


1.6 Feature 6 — Riesgo de mercado

In [12]:
# Por ahora simulamos riesgo medio (1)
df['market_risk'] = 1


2.1 Target 

In [13]:
# target = 1 si Future_Return_10d >= 2%
# target = 0 si Future_Return_10d <= 0%
df['target'] = np.nan

df.loc[df['Future_Return_10d'] >= 0.02, 'target'] = 1
df.loc[df['Future_Return_10d'] <= 0.0, 'target'] = 0

df = df.dropna(subset=['target'])

2.2 Matriz final

In [ ]:
features_simple = ['Price_Change_5d',
                   'price_vs_sma20',
                   'rsi_strength',
                   'Volume_Ratio',
                   'vol_level',
                   'market_risk']

X = df[features_simple]
y = df['target']


In [15]:
train_idx = []
test_idx = []

for tkr, g in df.groupby('Ticker'):
    g = g.sort_values('Date')
    split = int(len(g) * 0.8)
    train_idx.extend(g.index[:split])
    test_idx.extend(g.index[split:])


In [16]:
X_train = X.loc[train_idx]
y_train = y.loc[train_idx]

X_test = X.loc[test_idx]
y_test = y.loc[test_idx]


In [ ]:
scale_cols = ['Price_Change_5d',
              'price_vs_sma20',
              'Volume_Ratio']

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_indica = X_train.copy()
X_test_indica = X_test.copy()

X_train_indica[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_indica[scale_cols] = scaler.transform(X_test[scale_cols])


In [ ]:
simple_model = LogisticRegression(class_weight='balanced',
                                  max_iter=1000)

simple_model.fit(X_train_indica, y_train)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [ ]:
y_proba = simple_model.predict_proba(X_test_indica)[:,1]
y_pred = (y_proba > 0.5).astype(int)

classification_report(y_test, y_pred)


              precision    recall  f1-score   support

         0.0       0.55      0.55      0.55     56400
         1.0       0.47      0.47      0.47     48138

    accuracy                           0.52    104538
   macro avg       0.51      0.51      0.51    104538
weighted avg       0.52      0.52      0.52    104538



In [ ]:
coef = pd.Series(simple_model.coef_[0],
                 index=features_simple).sort_values()

coef


Price_Change_5d   -0.024934
rsi_strength      -0.010950
market_risk       -0.001213
price_vs_sma20     0.007620
vol_level          0.014888
Volume_Ratio       0.027061
dtype: float64

El modelo simplificado no pretende predecir con exactitud el comportamiento diario del mercado, sino ofrecer una estimación probabilística coherente y fácilmente interpretable que ayude a los usuarios a identificar escenarios potencialmente favorables.

Definición de rangos

Probabilidad	Nivel	Mensaje principal
< 40%	🔴 Baja	        No se recomienda operar
40–60%	🟡 Media	    Oportunidad moderada
> 60%	🟢 Alta	        Oportunidad interesante

Mensajes explicativos

Baja: La acción no muestra señales claras de oportunidad para mañana.
Se recomienda cautela o esperar mejores condiciones.

Media: La acción presenta algunas señales positivas, pero el escenario no es concluyente.
Puede ser interesante para perfiles con mayor tolerancia al riesgo.


Alta: La acción presenta algunas señales positivas, pero el escenario no es concluyente.
Puede ser interesante para perfiles con mayor tolerancia al riesgo.



Mensaje de riesgo de mercado
El contexto general del mercado es adverso.
Se recomienda extremar la gestión de riesgo o evitar nuevas posiciones.



La acción no muestra señales claras de oportunidad para mañana.
Se recomienda cautela o esperar mejores condiciones.


## conclusion

Las probabilidades generadas por el modelo se transforman en niveles de oportunidad y mensajes interpretables, con el objetivo de facilitar la toma de decisiones a usuarios no técnicos y evitar una falsa sensación de certeza.

In [ ]:
joblib.dump(simple_model, '../models/simple_model.pkl')


['../models/simple_model.pkl']

In [ ]:

joblib.dump(scaler, '../models/scaler.pkl')


['../models/scaler.pkl']